# Evaluation artifacts: tables and figures

This notebook reads the newest published backtest run under `evaluation/runs/` and rebuilds its
report's main tables and three figures.

A run published with its per-forecast table (`forecasts.csv`) is scored again here: every Brier
score, skill and reliability table is recomputed from that table with the project's own functions in
`pmfa.evaluation`, and checked against `metrics.json`. A run published as scores only has no
per-forecast table in the repository. Its tables and figures are drawn from `metrics.json`, and the
first cell says that nothing was recomputed.

It needs no API key and no network. Run it from the repository root or from `notebooks/`, in the
project environment with the notebook group (`uv sync --group notebook`).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

from pmfa.config import Config
from pmfa.evaluation.backtest import read_forecasts
from pmfa.evaluation.baselines import PIPELINE, SYSTEMS
from pmfa.evaluation.metrics import Reliability, reliability
from pmfa.evaluation.report import SYSTEM_COLORS, SYSTEM_LABELS
from pmfa.evaluation.run_metrics import forecast_block

ROOT = Path.cwd() if (Path.cwd() / "evaluation" / "runs").is_dir() else Path.cwd().parent
RUN_DIR = sorted(p for p in (ROOT / "evaluation" / "runs").iterdir() if p.is_dir())[-1]

manifest = json.loads((RUN_DIR / "manifest.json").read_text(encoding="utf-8"))
metrics = json.loads((RUN_DIR / "metrics.json").read_text(encoding="utf-8"))
config = Config.model_validate(manifest["config"])
HAS_ROWS = (RUN_DIR / "forecasts.csv").is_file()
rows = [row for row, _note in read_forecasts(RUN_DIR / "forecasts.csv")] if HAS_ROWS else []
scheduled = [r for r in rows if r.horizon_days is not None]
HORIZONS = [str(h) for h in sorted(config.evaluation.horizons_days, reverse=True)]

if HAS_ROWS:
    print(f"Run {RUN_DIR.name}: status {manifest['status']}, {len(rows)} forecast rows")
else:
    print(
        f"Run {RUN_DIR.name}: status {manifest['status']}, published as scores only. Its "
        "per-forecast table is not in the repository, so the numbers below are read from "
        "metrics.json and nothing is recomputed."
    )
for system, status in manifest["systems"].items():
    print(f"  {SYSTEM_LABELS.get(system, system)}: {status}")

## Tables

Brier score and Brier skill against the market, with 95% intervals that resample whole events. With
the per-forecast table they are recomputed from `forecasts.csv` with `forecast_block` (the function
that wrote `metrics.json`), and the cell stops if any recomputed number differs from `metrics.json`.
For a run published as scores only they are read from `metrics.json`.

In [ ]:
def interval(block):
    if block["estimate"] is None:
        return "not defined"
    if block["lower"] is None:
        return f"{block['estimate']:.4f} (interval not defined)"
    return f"{block['estimate']:.4f} [{block['lower']:.4f}, {block['upper']:.4f}]"


scores = {}
for system in SYSTEMS:
    if not HAS_ROWS:
        published = metrics["systems"].get(system, {}).get("by_horizon")
        if published:
            scores[system] = published
        continue
    mine = [r for r in scheduled if r.system == system]
    if not mine:
        continue
    scores[system] = {"all": forecast_block(mine, config)}
    for h in HORIZONS:
        at_h = [r for r in mine if r.horizon_days == int(h)]
        if at_h:
            scores[system][h] = forecast_block(at_h, config)

recomputed = scores if HAS_ROWS else {}
for system, by_horizon in recomputed.items():
    for horizon, block in by_horizon.items():
        assert block == metrics["systems"][system]["by_horizon"][horizon], (system, horizon)

lines = [
    "| System | Horizon | Forecasts | Markets | Events | Brier (answered) | Skill (answered) | Brier (abstentions as market) |",
    "|---|---|---|---|---|---|---|---|",
]
for system in SYSTEMS:
    if system not in scores:
        status = manifest["systems"].get(system, "not requested")
        lines.append(f"| {SYSTEM_LABELS[system]} | all | {status} | | | | | |")
        continue
    for horizon, block in scores[system].items():
        a, m = block["answered"], block["abstain_as_market"]
        lines.append(
            f"| {SYSTEM_LABELS[system]} | {horizon if horizon == 'all' else horizon + 'd'} "
            f"| {a['sample']['n_rows']} | {a['sample']['n_markets']} | {a['sample']['n_events']} "
            f"| {interval(a['brier'])} | {interval(a['skill'])} | {interval(m['brier'])} |"
        )
print("\n".join(lines))

In [ ]:
if HAS_ROWS:
    tables = {
        s: reliability(
            [r for r in scheduled if r.system == s], bins=config.evaluation.reliability_bins
        )
        for s in scores
    }
    for system, table in tables.items():
        assert table.model_dump(mode="json") == metrics["reliability"][system]["answered"], system
else:
    tables = {
        s: Reliability.model_validate(metrics["reliability"][s]["answered"])
        for s in scores
        if s in metrics["reliability"]
    }
for system, table in tables.items():
    # A scores-only run withholds the error when it would give back a withheld mean.
    ece = (
        f"{table.ece:.4f}"
        if table.ece is not None
        else "withheld"
        if any(b.count for b in table.bins)
        else "not defined"
    )
    print(f"{SYSTEM_LABELS[system]}, answered forecasts, all horizons (ECE {ece})")
    print("| Bin | Forecasts | Mean forecast | Share YES |")
    print("|---|---|---|---|")
    for b in table.bins:
        # A scores-only run withholds the mean forecast of a bin with few forecasts.
        mean = (
            "empty"
            if not b.count
            else "withheld"
            if b.mean_forecast is None
            else f"{b.mean_forecast:.3f}"
        )
        freq = "empty" if not b.count else f"{b.observed_frequency:.3f}"
        print(f"| [{b.lower:.1f}, {b.upper:.1f}) | {b.count} | {mean} | {freq} |")
    other = metrics["reliability"].get(system, {}).get("abstain_as_market")
    if other and "bins" in other:
        other_ece = "not defined" if other["ece"] is None else f"{other['ece']:.4f}"
        print(
            f"With abstentions scored as the market price ({other['n_rows']} forecasts) the ECE "
            f"is {other_ece}; that table is in metrics.json."
        )
    elif other:
        print(
            f"With abstentions scored as the market price ({other['n_rows']} forecasts): the "
            "table and its ECE are left out of a scores-only publication."
        )

## Figures

The report's three figures, drawn from the numbers above (the equity curve from `metrics.json`,
since the trades table is not committed).

In [ ]:
fig_brier, ax = plt.subplots(figsize=(7.5, 4.2), dpi=100)
width = 0.8 / max(len(scores), 1)
for i, system in enumerate(scores):
    xs, ys, err_lo, err_hi = [], [], [], []
    for j, h in enumerate(HORIZONS):
        if h not in scores[system]:
            continue
        brier = scores[system][h]["answered"]["brier"]
        if brier["estimate"] is None:
            continue  # the system answered nothing at this horizon
        xs.append(j + (i - (len(scores) - 1) / 2) * width)
        ys.append(brier["estimate"])
        err_lo.append(
            brier["estimate"]
            - (brier["lower"] if brier["lower"] is not None else brier["estimate"])
        )
        err_hi.append(
            (brier["upper"] if brier["upper"] is not None else brier["estimate"])
            - brier["estimate"]
        )
    ax.bar(xs, ys, width=width * 0.9, color=SYSTEM_COLORS[system], label=SYSTEM_LABELS[system])
    ax.errorbar(xs, ys, yerr=[err_lo, err_hi], fmt="none", ecolor="#666666", capsize=3)
ax.set_xticks(range(len(HORIZONS)), [f"{h} day(s) before close" for h in HORIZONS])
ax.set_ylabel("Brier score (lower is better)")
ax.set_title("Brier score by horizon, 95% interval resampling events")
ax.legend(frameon=False)
fig_brier.tight_layout()

In [ ]:
fig_reliability, ax = plt.subplots(figsize=(5.5, 5.2), dpi=100)
ax.plot([0, 1], [0, 1], color="#666666", linestyle="--", linewidth=1, label="Perfect calibration")
for system, table in tables.items():
    # A scores-only run withholds the mean forecast of a bin with few forecasts.
    points = [
        (b.mean_forecast, b.observed_frequency)
        for b in table.bins
        if b.count and b.mean_forecast is not None
    ]
    ax.plot(
        [p[0] for p in points],
        [p[1] for p in points],
        marker="o",
        linewidth=2,
        color=SYSTEM_COLORS[system],
        label=SYSTEM_LABELS[system],
    )
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Mean forecast in bin")
ax.set_ylabel("Share that resolved YES")
ax.set_title("Reliability (all horizons pooled)")
ax.legend(frameon=False, loc="upper left")
fig_reliability.tight_layout()

In [ ]:
from datetime import datetime

fig_equity, ax = plt.subplots(figsize=(7.5, 3.8), dpi=100)
curve = metrics["pnl"].get("equity_curve")  # left out of a scores-only publication
if curve:
    bankroll = float(metrics["pnl"]["bankroll"])
    times = [datetime.fromisoformat(p["at"]) for p in curve]
    equity = [float(p["equity"]) for p in curve]
    ax.step(
        [times[0], *times],
        [bankroll, *equity],
        where="post",
        color=SYSTEM_COLORS[PIPELINE],
        linewidth=2,
        label="Equity (open positions at cost)",
    )
    ax.axhline(bankroll, color="#666666", linestyle="--", linewidth=1, label="Not trading")
    ax.set_ylabel("Dollars")
    ax.set_title("Simulated equity after fees, news-triggered trading")
    ax.legend(frameon=False)
elif curve is None:
    ax.text(
        0.5,
        0.5,
        "The equity curve is left out of a scores-only publication.",
        ha="center",
        va="center",
    )
    ax.set_axis_off()
else:
    status = manifest["systems"].get(PIPELINE, "not requested")
    ax.text(0.5, 0.5, f"No simulated trades. Pipeline: {status}.", ha="center", va="center")
    ax.set_axis_off()
fig_equity.tight_layout()